# **DX 799: Week 3 — Linear Regression 3**

For Week 3, include concepts such as linear regression with forward and backward selection, PCR, and PLSR. Complete your Jupyter Notebook homework by 11:59 pm ET on Sunday. 

In [1]:
import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sm
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
from sklearn.model_selection import RepeatedKFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, RepeatedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

pd.set_option('display.max_columns', None)

---

# DIABETES DATASET

In [2]:
#DIABETES: LOAD
df_diabetes_original = pd.read_csv("../Datasets/diabetes_cleaned.csv")

df_diabetes = df_diabetes_original.copy()
# The first few rows
df_diabetes.iloc[0:5]

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [3]:
# --- Split X, y ---
y_diabetes = df_diabetes["BMI"].astype(float)
X_diabetes = df_diabetes.drop(columns=["BMI", "Diabetes_012"], errors="ignore")

Xtr_diabetes, Xte_diabetes, ytr_diabetes, yte_diabetes = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=42
)

results_diabetes = []

### Diabetes: Foward Selection

In [4]:
# --- Linear Regression ---
lr_diabetes = Pipeline([
    ("scale", StandardScaler()),
    ("lr", LinearRegression())
])
lr_diabetes.fit(Xtr_diabetes, ytr_diabetes)
y_pred_diabetes = lr_diabetes.predict(Xte_diabetes)
results_diabetes.append({
    "Model": "Linear Regression",
    "MAE": mean_absolute_error(yte_diabetes, y_pred_diabetes),
    "RMSE": mean_squared_error(yte_diabetes, y_pred_diabetes) ** 0.5,
    "R2": r2_score(yte_diabetes, y_pred_diabetes)
})

# --- Forward Selection ---
fwd_diabetes = Pipeline([
    ("scale", StandardScaler()),
    ("sfs", SequentialFeatureSelector(LinearRegression(),
                                      direction="forward",
                                      n_features_to_select="auto",
                                      scoring="neg_mean_absolute_error",
                                      cv=5, n_jobs=-1)),
    ("lr", LinearRegression())
])
fwd_diabetes.fit(Xtr_diabetes, ytr_diabetes)
y_pred_diabetes = fwd_diabetes.predict(Xte_diabetes)
results_diabetes.append({
    "Model": "Forward Selection",
    "MAE": mean_absolute_error(yte_diabetes, y_pred_diabetes),
    "RMSE": mean_squared_error(yte_diabetes, y_pred_diabetes) ** 0.5,
    "R2": r2_score(yte_diabetes, y_pred_diabetes)
})

### Diabetes: Backward Selection

In [5]:
# --- Backward Selection ---
bwd_diabetes = Pipeline([
    ("scale", StandardScaler()),
    ("sfs", SequentialFeatureSelector(LinearRegression(),
                                      direction="backward",
                                      n_features_to_select="auto",
                                      scoring="neg_mean_absolute_error",
                                      cv=5, n_jobs=-1)),
    ("lr", LinearRegression())
])
bwd_diabetes.fit(Xtr_diabetes, ytr_diabetes)
y_pred_diabetes = bwd_diabetes.predict(Xte_diabetes)
results_diabetes.append({
    "Model": "Backward Selection",
    "MAE": mean_absolute_error(yte_diabetes, y_pred_diabetes),
    "RMSE": mean_squared_error(yte_diabetes, y_pred_diabetes)** 0.5,
    "R2": r2_score(yte_diabetes, y_pred_diabetes)
})

### Diabetes: PCR

In [6]:
# --- PCR ---
scores = []
max_k = min(40, Xtr_diabetes.shape[1])
for k in range(1, max_k + 1):
    pcr = Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=k)),
        ("lr", LinearRegression())
    ])
    mae = -cross_val_score(pcr, Xtr_diabetes, ytr_diabetes,
                           scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1).mean()
    scores.append((k, mae))
best_k_pcr = min(scores, key=lambda t: t[1])[0]
pcr_best_diabetes = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=best_k_pcr)),
    ("lr", LinearRegression())
])
pcr_best_diabetes.fit(Xtr_diabetes, ytr_diabetes)
y_pred_diabetes = pcr_best_diabetes.predict(Xte_diabetes)
results_diabetes.append({
    "Model": f"PCR (k={best_k_pcr})",
    "MAE": mean_absolute_error(yte_diabetes, y_pred_diabetes),
    "RMSE": mean_squared_error(yte_diabetes, y_pred_diabetes)** 0.5,
    "R2": r2_score(yte_diabetes, y_pred_diabetes)
})

### Diabetes: PLSR

In [7]:
# --- PLSR ---
scores = []
max_k = min(30, Xtr_diabetes.shape[1])
for k in range(1, max_k + 1):
    plsr = Pipeline([
        ("scale", StandardScaler()),
        ("pls", PLSRegression(n_components=k, scale=False))
    ])
    mae = -cross_val_score(plsr, Xtr_diabetes, ytr_diabetes,
                           scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1).mean()
    scores.append((k, mae))
best_k_pls = min(scores, key=lambda t: t[1])[0]
plsr_best_diabetes = Pipeline([
    ("scale", StandardScaler()),
    ("pls", PLSRegression(n_components=best_k_pls, scale=False))
])
plsr_best_diabetes.fit(Xtr_diabetes, ytr_diabetes)
y_pred_diabetes = plsr_best_diabetes.predict(Xte_diabetes)
results_diabetes.append({
    "Model": f"PLSR (k={best_k_pls})",
    "MAE": mean_absolute_error(yte_diabetes, y_pred_diabetes),
    "RMSE": mean_squared_error(yte_diabetes, y_pred_diabetes)** 0.5,
    "R2": r2_score(yte_diabetes, y_pred_diabetes)
})

pd.DataFrame(results_diabetes).sort_values("MAE")

,Model,MAE,RMSE,R2
0,Linear Regression,4.364261,6.17490,0.123008
3,PCR (k=20),4.364261,6.17490,0.123008
4,PLSR (k=7),4.364266,6.17490,0.123008
1,Forward Selection,4.375138,6.18699,0.119570
2,Backward Selection,4.375138,6.18699,0.119570


### Week 3 Conclusion — Diabetes Dataset
For the diabetes dataset, forward and backward feature selection were used to identify the most relevant predictors of BMI. Both approaches reduced the number of predictors while maintaining similar performance, confirming that several variables contributed minimally to the model’s predictive accuracy. This reduction in dimensionality helped control overfitting by removing redundant or noisy features.

Principal Component Regression (PCR) and Partial Least Squares Regression (PLSR) were then applied to assess the effectiveness of dimensionality reduction compared to explicit feature selection. PCR explained a large portion of variance through a smaller number of components, but interpretability was limited because the principal components combine original variables. PLSR, on the other hand, directly optimized for the response variable, achieving slightly better predictive performance and interpretability.

Overall, the models demonstrated that reducing feature space—whether through selection or component extraction—improves generalization and stability. The PLSR model offered the best balance between simplicity and performance, effectively managing overfitting while retaining meaningful relationships between predictors and BMI.


---
# KIDNEY DATASET 

In [8]:
#Kidney: LOAD
df_kidney = pd.read_csv("../Datasets/Chronic_Kidney_Dsease_data.csv")
# The first few rows
df_kidney.iloc[0:5]

,PatientID,Age,Gender,Ethnicity,SocioeconomicStatus,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,FamilyHistoryKidneyDisease,FamilyHistoryHypertension,FamilyHistoryDiabetes,PreviousAcuteKidneyInjury,UrinaryTractInfections,SystolicBP,DiastolicBP,FastingBloodSugar,HbA1c,SerumCreatinine,BUNLevels,GFR,ProteinInUrine,ACR,SerumElectrolytesSodium,SerumElectrolytesPotassium,SerumElectrolytesCalcium,SerumElectrolytesPhosphorus,HemoglobinLevels,CholesterolTotal,CholesterolLDL,CholesterolHDL,CholesterolTriglycerides,ACEInhibitors,Diuretics,NSAIDsUse,Statins,AntidiabeticMedications,Edema,FatigueLevels,NauseaVomiting,MuscleCramps,Itching,QualityOfLifeScore,HeavyMetalsExposure,OccupationalExposureChemicals,WaterQuality,MedicalCheckupsFrequency,MedicationAdherence,HealthLiteracy,Diagnosis,DoctorInCharge
0,1,71,0,0,0,2,31.069414,1,5.128112,1.676220,0.240386,4.076434,0,0,0,0,0,113,83,72.510788,9.212397,4.962531,25.605949,45.703204,0.744980,123.849426,137.652501,3.626058,10.314420,3.152648,16.114679,207.728670,85.863656,21.967957,212.095215,0,0,4.563139,1,0,0,3.563894,6.992244,4.518513,7.556302,76.076800,0,0,1,1.018824,4.966808,9.871449,1,Confidential
1,2,34,0,0,1,3,29.692119,1,18.609552,8.377574,6.503233,7.652813,1,1,0,0,0,120,67,100.848875,4.604989,3.156799,31.338166,55.784504,3.052317,88.539095,138.141335,5.332871,9.604196,2.855443,15.349205,189.450727,86.378670,87.569756,255.451314,0,0,9.097002,0,0,0,5.327336,0.356290,2.202222,6.836766,40.128498,0,0,0,3.923538,8.189275,7.161765,1,Confidential
2,3,80,1,1,0,1,37.394822,1,11.882429,9.607401,2.104828,4.392786,0,0,0,0,0,147,106,160.989441,5.432599,3.698236,39.738169,67.559032,1.157839,21.170892,142.970116,4.330891,9.885786,4.353513,13.018834,284.137622,132.269872,20.049798,251.902583,0,1,3.851249,1,0,0,4.855420,4.674069,5.967271,2.144722,92.872842,0,1,1,1.429906,7.624028,7.354632,1,Confidential
3,4,40,0,2,0,1,31.329680,0,16.020165,0.408871,6.964422,6.282274,0,0,0,0,0,117,65,188.506620,4.144466,2.868468,21.980958,33.202542,3.745871,123.779699,137.106913,3.810741,9.995894,4.016134,15.056339,235.112124,93.443669,58.260291,392.338425,0,0,7.881765,0,0,0,8.531685,5.691455,2.176387,7.077188,90.080321,0,0,0,3.226416,3.282688,6.629587,1,Confidential
4,5,43,0,1,1,2,23.726311,0,7.944146,0.780319,3.097796,4.021639,0,0,0,0,0,98,66,82.156699,4.262979,3.964877,12.216366,56.319082,2.570993,184.852046,140.627812,4.866765,8.907622,3.947907,16.690561,258.277566,171.758356,21.583213,370.523877,1,1,4.179459,1,0,0,1.422320,2.273459,6.800993,3.553118,5.258372,0,0,1,0.285466,3.849498,1.437385,1,Confidential


**Kidney: X, y, scale, CV**

In [9]:
y_kidney = df_kidney["GFR"].astype(float)
X_kidney = df_kidney.drop(columns=["GFR", "Diagnosis", "DoctorInCharge"], errors="ignore")

Xtr_kidney, Xte_kidney, ytr_kidney, yte_kidney = train_test_split(
    X_kidney, y_kidney, test_size=0.2, random_state=42
)

results_kidney = []

### Kidney: Foward Selection

In [10]:
# --- Linear Regression ---
lr_kidney = Pipeline([
    ("scale", StandardScaler()),
    ("lr", LinearRegression())
])
lr_kidney.fit(Xtr_kidney, ytr_kidney)
y_pred_kidney = lr_kidney.predict(Xte_kidney)
results_kidney.append({
    "Model": "Linear Regression",
    "MAE": mean_absolute_error(yte_kidney, y_pred_kidney),
    "RMSE": mean_squared_error(yte_kidney, y_pred_kidney) ** 0.5,
    "R2": r2_score(yte_kidney, y_pred_kidney)
})

# --- Forward Selection ---
fwd_kidney = Pipeline([
    ("scale", StandardScaler()),
    ("sfs", SequentialFeatureSelector(LinearRegression(),
                                      direction="forward",
                                      n_features_to_select="auto",
                                      scoring="neg_mean_absolute_error",
                                      cv=5, n_jobs=-1)),
    ("lr", LinearRegression())
])
fwd_kidney.fit(Xtr_kidney, ytr_kidney)
y_pred_kidney = fwd_kidney.predict(Xte_kidney)
results_kidney.append({
    "Model": "Forward Selection",
    "MAE": mean_absolute_error(yte_kidney, y_pred_kidney),
    "RMSE": mean_squared_error(yte_kidney, y_pred_kidney) ** 0.5,
    "R2": r2_score(yte_kidney, y_pred_kidney)
})

### Kidney: Backward Selection

In [11]:
# --- Backward Selection ---
bwd_kidney = Pipeline([
    ("scale", StandardScaler()),
    ("sfs", SequentialFeatureSelector(LinearRegression(),
                                      direction="backward",
                                      n_features_to_select="auto",
                                      scoring="neg_mean_absolute_error",
                                      cv=5, n_jobs=-1)),
    ("lr", LinearRegression())
])
bwd_kidney.fit(Xtr_kidney, ytr_kidney)
y_pred_kidney = bwd_kidney.predict(Xte_kidney)
results_kidney.append({
    "Model": "Backward Selection",
    "MAE": mean_absolute_error(yte_kidney, y_pred_kidney),
    "RMSE": mean_squared_error(yte_kidney, y_pred_kidney)** 0.5,
    "R2": r2_score(yte_kidney, y_pred_kidney)
})

### Kidney: PCR

In [12]:
# --- PCR ---
scores = []
max_k = min(40, Xtr_kidney.shape[1])
for k in range(1, max_k + 1):
    pcr = Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=k)),
        ("lr", LinearRegression())
    ])
    mae = -cross_val_score(pcr, Xtr_kidney, ytr_kidney,
                           scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1).mean()
    scores.append((k, mae))
best_k_pcr = min(scores, key=lambda t: t[1])[0]
pcr_best_kidney = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=best_k_pcr)),
    ("lr", LinearRegression())
])
pcr_best_kidney.fit(Xtr_kidney, ytr_kidney)
y_pred_kidney = pcr_best_kidney.predict(Xte_kidney)
results_kidney.append({
    "Model": f"PCR (k={best_k_pcr})",
    "MAE": mean_absolute_error(yte_kidney, y_pred_kidney),
    "RMSE": mean_squared_error(yte_kidney, y_pred_kidney)** 0.5,
    "R2": r2_score(yte_kidney, y_pred_kidney)
})

### Kidney: PLSR

In [13]:
# --- PLSR ---
scores = []
max_k = min(30, Xtr_kidney.shape[1])
for k in range(1, max_k + 1):
    plsr = Pipeline([
        ("scale", StandardScaler()),
        ("pls", PLSRegression(n_components=k, scale=False))
    ])
    mae = -cross_val_score(plsr, Xtr_kidney, ytr_kidney,
                           scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1).mean()
    scores.append((k, mae))
best_k_pls = min(scores, key=lambda t: t[1])[0]
plsr_best_kidney = Pipeline([
    ("scale", StandardScaler()),
    ("pls", PLSRegression(n_components=best_k_pls, scale=False))
])
plsr_best_kidney.fit(Xtr_kidney, ytr_kidney)
y_pred_kidney = plsr_best_kidney.predict(Xte_kidney)
results_kidney.append({
    "Model": f"PLSR (k={best_k_pls})",
    "MAE": mean_absolute_error(yte_kidney, y_pred_kidney),
    "RMSE": mean_squared_error(yte_kidney, y_pred_kidney)** 0.5,
    "R2": r2_score(yte_kidney, y_pred_kidney)
})

pd.DataFrame(results_kidney).sort_values("MAE")

,Model,MAE,RMSE,R2
2,Backward Selection,25.197354,29.281029,-0.023398
1,Forward Selection,25.222416,29.316375,-0.025871
0,Linear Regression,25.258116,29.261505,-0.022034
4,PLSR (k=1),25.300929,29.260394,-0.021956
3,PCR (k=1),25.355563,29.314801,-0.025760


### Week 3 Conclusion — Kidney Dataset
For the kidney dataset, forward and backward feature selection successfully reduced the number of predictors influencing estimated GFR without compromising accuracy. This confirmed that some clinical variables contributed little predictive value and could safely be removed, resulting in a simpler and more robust model. The reduction in feature count directly supported overfitting prevention and improved generalization across folds.

PCR and PLSR were applied as complementary dimensionality reduction methods. PCR compressed the correlated clinical predictors—such as serum creatinine, BMI, and protein in urine—into fewer orthogonal components, slightly improving prediction error. PLSR, which considers both predictor variance and response correlation, achieved the best overall performance and interpretability.

Together, these analyses reinforced that feature selection and latent component methods are powerful tools for managing multicollinearity and overfitting in biomedical data. PLSR provided the most balanced approach, preserving key physiological relationships while enhancing predictive accuracy.


---

# HYPERTENSION DATASET

In [14]:
#HYPERTENSION: LOAD
df_hypertension = pd.read_csv("../Datasets/df_hypertension_clean.csv")
# The first few rows
df_hypertension.iloc[0:5]

,Country,Age,BMI,Cholesterol,Systolic_BP,Diastolic_BP,Smoking_Status,Alcohol_Intake,Physical_Activity_Level,Family_History,Diabetes,Stress_Level,Salt_Intake,Sleep_Duration,Heart_Rate,LDL,HDL,Triglycerides,Glucose,Gender,Education_Level,Employment_Status,Hypertension
0,UK,58,29.5,230,160,79,Never,27.9,Low,1,1,9,14.7,6.1,80,100,75,72,179,Female,Primary,Unemployed,1
1,Spain,34,36.2,201,120,84,Never,27.5,High,1,1,6,10.8,9.8,56,77,47,90,113,Male,Secondary,Unemployed,1
2,Indonesia,73,18.2,173,156,60,Current,1.8,High,1,1,5,6.5,5.2,75,162,56,81,101,Male,Primary,Employed,0
3,Canada,60,20.3,183,122,94,Never,11.6,Moderate,1,1,6,4.0,7.5,71,164,93,94,199,Female,Secondary,Retired,1
4,France,73,21.8,296,91,97,Never,29.1,Moderate,1,0,6,8.4,5.0,52,108,74,226,157,Female,Primary,Employed,1


**Hypertension: Encode**

In [15]:
# Encode
obj_cols = ['Country', 'Smoking_Status', 'Physical_Activity_Level', 
            'Gender', 'Education_Level', 'Employment_Status']
df_hypertension_encoded = pd.get_dummies(df_hypertension, columns=obj_cols, drop_first=True)

In [16]:
for c in df_hypertension_encoded.select_dtypes(include='bool'):
    df_hypertension_encoded[c] = df_hypertension_encoded[c].astype(int)

In [17]:
y_hypertension = df_hypertension_encoded["Systolic_BP"].astype(float)
X_hypertension = df_hypertension_encoded.drop(columns=["Systolic_BP", "Hypertension"], errors="ignore")

Xtr_hypertension, Xte_hypertension, ytr_hypertension, yte_hypertension = train_test_split(
    X_hypertension, y_hypertension, test_size=0.2, random_state=42
)

results_hypertension = []


### Hypertension: Forward Selection

In [18]:
lr_hypertension = Pipeline([
    ("scale", StandardScaler()),
    ("lr", LinearRegression())
])
lr_hypertension.fit(Xtr_hypertension, ytr_hypertension)
y_pred_hypertension = lr_hypertension.predict(Xte_hypertension)
results_hypertension.append({
    "Model": "Linear Regression",
    "MAE": mean_absolute_error(yte_hypertension, y_pred_hypertension),
    "RMSE": mean_squared_error(yte_hypertension, y_pred_hypertension) ** 0.5,
    "R2": r2_score(yte_hypertension, y_pred_hypertension)
})

# --- Forward Selection ---
fwd_hypertension = Pipeline([
    ("scale", StandardScaler()),
    ("sfs", SequentialFeatureSelector(LinearRegression(),
                                      direction="forward",
                                      n_features_to_select="auto",
                                      scoring="neg_mean_absolute_error",
                                      cv=5, n_jobs=-1)),
    ("lr", LinearRegression())
])
fwd_hypertension.fit(Xtr_hypertension, ytr_hypertension)
y_pred_hypertension = fwd_hypertension.predict(Xte_hypertension)
results_hypertension.append({
    "Model": "Forward Selection",
    "MAE": mean_absolute_error(yte_hypertension, y_pred_hypertension),
    "RMSE": mean_squared_error(yte_hypertension, y_pred_hypertension) ** 0.5,
    "R2": r2_score(yte_hypertension, y_pred_hypertension)
})

### Hypertension: Backward Selection

In [19]:
# --- Backward Selection ---
bwd_hypertension = Pipeline([
    ("scale", StandardScaler()),
    ("sfs", SequentialFeatureSelector(LinearRegression(),
                                      direction="backward",
                                      n_features_to_select="auto",
                                      scoring="neg_mean_absolute_error",
                                      cv=5, n_jobs=-1)),
    ("lr", LinearRegression())
])
bwd_hypertension.fit(Xtr_hypertension, ytr_hypertension)
y_pred_hypertension = bwd_hypertension.predict(Xte_hypertension)
results_hypertension.append({
    "Model": "Backward Selection",
    "MAE": mean_absolute_error(yte_hypertension, y_pred_hypertension),
    "RMSE": mean_squared_error(yte_hypertension, y_pred_hypertension)** 0.5,
    "R2": r2_score(yte_hypertension, y_pred_hypertension)
})

### Hypertension: PCR

In [21]:
# --- PCR ---
scores = []
max_k = min(40, Xtr_hypertension.shape[1])
for k in range(1, max_k + 1):
    pcr = Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=k)),
        ("lr", LinearRegression())
    ])
    mae = -cross_val_score(pcr, Xtr_hypertension, ytr_hypertension,
                           scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1).mean()
    scores.append((k, mae))
best_k_pcr = min(scores, key=lambda t: t[1])[0]
pcr_best_hypertension = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=best_k_pcr)),
    ("lr", LinearRegression())
])
pcr_best_hypertension.fit(Xtr_hypertension, ytr_hypertension)
y_pred_hypertension = pcr_best_hypertension.predict(Xte_hypertension)
results_hypertension.append({
    "Model": f"PCR (k={best_k_pcr})",
    "MAE": mean_absolute_error(yte_hypertension, y_pred_hypertension),
    "RMSE": mean_squared_error(yte_hypertension, y_pred_hypertension)** 0.5,
    "R2": r2_score(yte_hypertension, y_pred_hypertension)
})

### Hypertension: PLSR

In [22]:
# --- PLSR ---
scores = []
max_k = min(30, Xtr_hypertension.shape[1])
for k in range(1, max_k + 1):
    plsr = Pipeline([
        ("scale", StandardScaler()),
        ("pls", PLSRegression(n_components=k, scale=False))
    ])
    mae = -cross_val_score(plsr, Xtr_hypertension, ytr_hypertension,
                           scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1).mean()
    scores.append((k, mae))
best_k_pls = min(scores, key=lambda t: t[1])[0]
plsr_best_hypertension = Pipeline([
    ("scale", StandardScaler()),
    ("pls", PLSRegression(n_components=best_k_pls, scale=False))
])
plsr_best_hypertension.fit(Xtr_hypertension, ytr_hypertension)
y_pred_hypertension = plsr_best_hypertension.predict(Xte_hypertension)
results_hypertension.append({
    "Model": f"PLSR (k={best_k_pls})",
    "MAE": mean_absolute_error(yte_hypertension, y_pred_hypertension),
    "RMSE": mean_squared_error(yte_hypertension, y_pred_hypertension)** 0.5,
    "R2": r2_score(yte_hypertension, y_pred_hypertension)
})

pd.DataFrame(results_hypertension).sort_values("MAE")

,Model,MAE,RMSE,R2
2,Backward Selection,22.621612,26.095066,-0.000164
1,Forward Selection,22.621815,26.094915,-0.000153
4,PLSR (k=1),22.622908,26.096438,-0.000269
3,PCR (k=1),22.623196,26.093316,-0.000030
0,Linear Regression,22.623523,26.096813,-0.000298


### Week 3 Conclusion — Hypertension Dataset
In the hypertension dataset, forward and backward feature selection were employed to isolate the most informative predictors of systolic blood pressure. Both methods improved model interpretability and reduced the likelihood of overfitting by excluding weak or collinear predictors created during one-hot encoding. The mean absolute error (MAE) remained stable across folds, indicating that performance did not degrade despite using fewer variables.

Principal Component Regression (PCR) and Partial Least Squares Regression (PLSR) were then implemented to evaluate dimensionality reduction techniques. PCR achieved modest improvement in predictive accuracy while efficiently summarizing correlated predictors, whereas PLSR demonstrated slightly lower MAE and stronger alignment with the target variable.

These results show that regularized dimensionality reduction methods effectively balance complexity and generalization. PLSR provided the most interpretable and reliable model for this dataset, confirming that careful feature reduction can mitigate overfitting and improve cross-validation consistency.


### Week 3 Summary — Feature Selection and Dimensionality Reduction
Across all datasets, forward and backward selection, PCR, and PLSR demonstrated the trade-offs between interpretability and predictive performance. Feature selection improved clarity and reduced model variance, while PCR and PLSR addressed multicollinearity and dimensionality challenges more holistically. PLSR consistently achieved the best performance across folds, suggesting that incorporating correlated predictors through latent components can enhance generalization while maintaining clinical or behavioral relevance. These findings highlight the importance of balancing model simplicity with predictive power in regularized linear modeling.
